In [1]:
!pip install -U transformers>=4.48.0
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 6.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login

# Colab のシークレット環境変数から Hugging Face トークンを取得
huggingface_token = userdata.get("HUGGINGFACE_TOKEN")

if huggingface_token:
    login(huggingface_token)
    print("ログイン成功")
else:
    print("HUGGINGFACE_TOKEN が設定されていません。")


ログイン成功


In [3]:
from transformers import AutoModel, AutoTokenizer
import os
import sys
import torch
import random
from datasets import load_dataset, Dataset
from tokenizers import BertWordPieceTokenizer
from transformers import ModernBertConfig, ModernBertForMaskedLM, PreTrainedTokenizerFast, DataCollatorForLanguageModeling, Trainer, TrainingArguments


# モデル名
repo_name = "Shuu12121/CodeMorph-ModernBERT"
# Hugging Face からモデルをロード
model = ModernBertForMaskedLM.from_pretrained(repo_name)
tokenizer = AutoTokenizer.from_pretrained(repo_name)

print("モデルのロード成功！")
print(model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/408k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
     

ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
      (1-11): 11

In [4]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)
print(fill_mask("def add_numbers(a, b): return a + [MASK]"))

Device set to use cuda:0


[{'score': 0.9981516003608704, 'token': 49908, 'token_str': 'b', 'sequence': 'def add _ numbers ( a, b ) : return a + b'}, {'score': 0.0010269464692100883, 'token': 49886, 'token_str': 'a', 'sequence': 'def add _ numbers ( a, b ) : return a + a'}, {'score': 0.00034104351652786136, 'token': 49891, 'token_str': 'c', 'sequence': 'def add _ numbers ( a, b ) : return a + c'}, {'score': 8.821619121590629e-05, 'token': 49938, 'token_str': '1', 'sequence': 'def add _ numbers ( a, b ) : return a + 1'}, {'score': 2.8221440516063012e-05, 'token': 159, 'token_str': 'end', 'sequence': 'def add _ numbers ( a, b ) : return a + end'}]


In [5]:
import torch

def get_embedding(text, model, tokenizer, device="cuda"):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    # token_type_ids があれば削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding

embedding = get_embedding("def my_function(): pass", model, tokenizer)
print(embedding.shape)


torch.Size([1, 768])


In [7]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデル・各言語でコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    # Python の乱数シードを設定（候補サンプルの抽出などに影響します）
    random.seed(35)
    np.random.seed(35)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットの言語一覧（例）
    # ------------------------------
    languages = ["python", "java", "javascript", "php", "ruby", "go"]
    max_examples = 1000  # 各言語ごとに先頭100サンプルを利用（ランダムサンプリング後）

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-BPE-1.0", "class": AutoModelForMaskedLM},
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/codebert-base-mlm", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
        {"name": "Shuu12121/CodeHawks-ModernBERT-preview", "class": AutoModelForMaskedLM},
    ]

    for lang in languages:
        print(f"\n==== 言語: {lang} の評価を開始 ====")
        # データセットをロード後、シャッフルしてランダムサンプルを抽出
        dataset = load_dataset("google/code_x_glue_ct_code_to_text", lang, split="test", trust_remote_code=True)
        dataset = dataset.shuffle(seed=35)
        subset = dataset.select(range(max_examples))

        for config in model_configs:
            model_name = config["name"]
            model_class = config["class"]
            print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = model_class.from_pretrained(model_name)
            model.to(device)

            metrics = evaluate_code_search(model, tokenizer, subset, device,
                                           max_examples=max_examples,
                                           pool_size=100,
                                           query_field="docstring",
                                           code_field="code")
            display_code_search_results(metrics, f"{model_name} - {lang}")


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base-mlm - python Code Search Evaluation Results ====
MRR:         0.5243
MAP:         0.5243
R-Precision: 0.4480

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4480    0.4480    0.4480    0.4480    0.4480         0.4480         
5     0.5860    0.4986    0.5204    0.5206    0.5860         0.5860         
10    0.6660    0.5092    0.5462    0.5393    0.6660         0.6660         
50    0.9450    0.5234    0.6094    0.5662    0.9450         0.9450         
100   1.0000    0.5243    0.6184    0.5678    1.0000         1.0000         

Salesforce/codet5p-220m-bimodal を評価します (候補プールサイズ: 100)...


You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - python Code Search Evaluation Results ====
MRR:         0.8266
MAP:         0.8266
R-Precision: 0.7610

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.7610    0.7610    0.7610    0.7610    0.7610         0.7610         
5     0.9110    0.8191    0.8422    0.8433    0.9110         0.9110         
10    0.9470    0.8244    0.8543    0.8525    0.9470         0.9470         
50    0.9880    0.8264    0.8635    0.8563    0.9880         0.9880         
100   1.0000    0.8266    0.8655    0.8567    1.0000         1.0000         

Shuu12121/CodeHawks-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeHawks-ModernBERT - python Code Search Evaluation Results ====
MRR:         0.8551
MAP:         0.8551

...


Shuu12121/CodeMorph-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT - go Code Search Evaluation Results ====
MRR:         0.5423
MAP:         0.5423
R-Precision: 0.4420

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4420    0.4420    0.4420    0.4420    0.4420         0.4420         
5     0.6440    0.5191    0.5503    0.5513    0.6440         0.6440         
10    0.7260    0.5302    0.5770    0.5708    0.7260         0.7260         
50    0.9480    0.5415    0.6272    0.5921    0.9480         0.9480         
100   1.0000    0.5423    0.6357    0.5936    1.0000         1.0000         

Shuu12121/CodeMorph-ModernBERT-BPE-1.0 を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT-BPE-1.0 - go Code Searc

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base-mlm - go Code Search Evaluation Results ====
MRR:         0.3262
MAP:         0.3262
R-Precision: 0.2440

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2440    0.2440    0.2440    0.2440    0.2440         0.2440         
5     0.3870    0.2951    0.3179    0.3176    0.3870         0.3870         
10    0.4780    0.3071    0.3471    0.3387    0.4780         0.4780         
50    0.8380    0.3237    0.4262    0.3703    0.8380         0.8380         
100   1.0000    0.3262    0.4528    0.3751    1.0000         1.0000         

Salesforce/codet5p-220m-bimodal を評価します (候補プールサイズ: 100)...


You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - go Code Search Evaluation Results ====
MRR:         0.8117
MAP:         0.8117
R-Precision: 0.7280

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.7280    0.7280    0.7280    0.7280    0.7280         0.7280         
5     0.9170    0.8040    0.8325    0.8345    0.9170         0.9170         
10    0.9560    0.8093    0.8452    0.8438    0.9560         0.9560         
50    0.9970    0.8117    0.8549    0.8482    0.9970         0.9970         
100   1.0000    0.8117    0.8553    0.8483    1.0000         1.0000         

Shuu12121/CodeHawks-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeHawks-ModernBERT - go Code Search Evaluation Results ====
MRR:         0.9043
MAP:         0.9043
R-Preci

In [8]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデル・各言語でコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    # Python の乱数シードを設定（候補サンプルの抽出などに影響します）
    random.seed(35)
    np.random.seed(35)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットの言語一覧（例）
    # ------------------------------
    languages = ["python", "java", "javascript", "php", "ruby", "go"]
    max_examples = 1000  # 各言語ごとに先頭100サンプルを利用（ランダムサンプリング後）

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-BPE-1.0", "class": AutoModelForMaskedLM},
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/codebert-base-mlm", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
        {"name": "Shuu12121/CodeHawks-ModernBERT-preview", "class": AutoModelForMaskedLM},
    ]

    for lang in languages:
        print(f"\n==== 言語: {lang} の評価を開始 ====")
        # データセットをロード後、シャッフルしてランダムサンプルを抽出
        dataset = load_dataset("code-search-net/code_search_net", lang, split="test", trust_remote_code=True)
        dataset = dataset.shuffle(seed=35)
        subset = dataset.select(range(max_examples))

        for config in model_configs:
            model_name = config["name"]
            model_class = config["class"]
            print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = model_class.from_pretrained(model_name)
            model.to(device)

            metrics = evaluate_code_search(model, tokenizer, subset, device,
                                           max_examples=max_examples,
                                           pool_size=100,
                                           query_field="func_documentation_string",
                                           code_field="func_code_string")
            display_code_search_results(metrics, f"{model_name} - {lang}")


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

README.md:   0%|          | 0.00/12.9k [00:00<?, ?B/s]

code_search_net.py:   0%|          | 0.00/8.44k [00:00<?, ?B/s]

python.zip:   0%|          | 0.00/941M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

...


Shuu12121/CodeMorph-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT - go Code Search Evaluation Results ====
MRR:         0.5028
MAP:         0.5028
R-Precision: 0.4030

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4030    0.4030    0.4030    0.4030    0.4030         0.4030         
5     0.6110    0.4812    0.5137    0.5145    0.6110         0.6110         
10    0.6760    0.4901    0.5348    0.5300    0.6760         0.6760         
50    0.9060    0.5013    0.5862    0.5512    0.9060         0.9060         
100   1.0000    0.5028    0.6020    0.5543    1.0000         1.0000         

Shuu12121/CodeMorph-ModernBERT-BPE-1.0 を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT-BPE-1.0 - go Code Searc

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base-mlm - go Code Search Evaluation Results ====
MRR:         0.3302
MAP:         0.3302
R-Precision: 0.2320

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2320    0.2320    0.2320    0.2320    0.2320         0.2320         
5     0.4160    0.2971    0.3266    0.3261    0.4160         0.4160         
10    0.5410    0.3137    0.3669    0.3553    0.5410         0.5410         
50    0.8330    0.3277    0.4316    0.3818    0.8330         0.8330         
100   1.0000    0.3302    0.4589    0.3866    1.0000         1.0000         

Salesforce/codet5p-220m-bimodal を評価します (候補プールサイズ: 100)...


You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - go Code Search Evaluation Results ====
MRR:         0.7478
MAP:         0.7478
R-Precision: 0.6480

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.6480    0.6480    0.6480    0.6480    0.6480         0.6480         
5     0.8810    0.7358    0.7720    0.7730    0.8810         0.8810         
10    0.9480    0.7451    0.7941    0.7893    0.9480         0.9480         
50    0.9970    0.7477    0.8055    0.7943    0.9970         0.9970         
100   1.0000    0.7478    0.8060    0.7944    1.0000         1.0000         

Shuu12121/CodeHawks-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeHawks-ModernBERT - go Code Search Evaluation Results ====
MRR:         0.8297
MAP:         0.8297
R-Preci

In [11]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデルでコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットのロード
    # ------------------------------
    print("\ngoogle/code_x_glue_tc_nl_code_search_adv データセット (Test) をロードします...")
    tc_dataset = load_dataset("google/code_x_glue_tc_nl_code_search_adv", split="test", trust_remote_code=True)
    dataset = tc_dataset.shuffle(seed=35)
    max_examples= 19210
    subset = dataset.select(range(max_examples))

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "Shuu12121/CodeHawks-ModernBERT-preview", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
    ]

    for config in model_configs:
        model_name = config["name"]
        model_class = config["class"]
        print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name)
        model.to(device)

        metrics = evaluate_code_search(model, tokenizer, subset, device,
                                       max_examples=19210,
                                       pool_size=100,
                                       query_field="docstring",
                                       code_field="code")
        display_code_search_results(metrics, model_name)


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

tokenizer_config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/804k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/462k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeHawks-ModernBERT-preview Code Search Evaluation Results ====
MRR:         0.6679
MAP:         0.6679
R-Precision: 0.5748

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.5748    0.5748    0.5748    0.5748    0.5748         0.5748         
5     0.7745    0.6518    0.6825    0.6838    0.7745         0.7745         
10    0.8409    0.6608    0.7041    0.6995    0.8409         0.8409         
50    0.9710    0.6675    0.7336    0.7122    0.9710         0.9710         
100   1.0000    0.6679    0.7384    0.7130    1.0000         1.0000         

Salesforce/codet5p-220m-bimodal を評価します (候補プールサイズ: 100)...


You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal Code Search Evaluation Results ====
MRR:         0.5359
MAP:         0.5359
R-Precision: 0.4258

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4258    0.4258    0.4258    0.4258    0.4258         0.4258         
5     0.6581    0.5129    0.5491    0.5499    0.6581         0.6581         
10    0.7526    0.5256    0.5798    0.5722    0.7526         0.7526         
50    0.9457    0.5351    0.6230    0.5902    0.9457         0.9457         
100   1.0000    0.5359    0.6319    0.5919    1.0000         1.0000         


In [12]:
# prompt: 切断する

from google.colab import runtime
runtime.unassign()
